# オプション課題：ViT はゼロから学習して CNN に勝てるか

**問い**：画像を「パッチの列」として自己注意で見る ViT（Vision Transformer）は，自分の少ない写真で，MLP・CNN に勝てるか．

- 第2回の演習2 で見た自己注意（Transformer）を画像に使う．CNN と違い「近くの画素どうしが関係する」という前提を持たない
- 演習2 と同じシード 0〜4・20 エポックで，MLP・CNN・ViT を比べる．提出は任意
- 第5回（転移学習）では，大量の画像で学習済みの ViT を使う

## (0) 番号の設定

In [ ]:
# ===== (0) 番号の設定 =====
MEMBER_NO = 2                 # ← グループ内の自分の番号 (1〜5)．presenter はこの番号の順に演習ごとに交代する

# 授業用フォルダ（AI_TD）のルートを import パスに追加する（ノートブックをどこから開いても動く）
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".dlcourse_root").exists())
sys.path.insert(0, str(ROOT))
print("作業フォルダ:", ROOT)

## (1) 予測（実行する前に書く）

このセルをダブルクリックして書き込む．

- 3 つのテスト正解率（5 回の平均）の順位：＿＿ ＞ ＿＿ ＞ ＿＿

## (2) ViT のモデル（コードテンプレート⑦の代わり）

In [ ]:
import torch
import torch.nn as nn


class TinyViT(nn.Module):
    """小さな ViT（Vision Transformer）．画像を 8×8 のパッチに切り，パッチを単語のように並べて自己注意で見る．"""

    def __init__(self, n_class, image_size=64, patch=8, dim=64, depth=2, heads=4):
        super().__init__()
        n_patch = (image_size // patch) ** 2                                      # 64×64 なら 8×8 = 64 個
        self.to_patch = nn.Conv2d(3, dim, kernel_size=patch, stride=patch)        # 各パッチを dim 次元のベクトルにする
        self.cls = nn.Parameter(torch.zeros(1, 1, dim))                           # 分類用の印（最後にこれの出力で分類する）
        self.pos = nn.Parameter(torch.randn(1, n_patch + 1, dim) * 0.02)          # 位置の情報（どこのパッチか）
        layer = nn.TransformerEncoderLayer(dim, heads, dim_feedforward=dim * 2, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, depth, enable_nested_tensor=False)   # 自己注意の層を depth 段
        self.head = nn.Linear(dim, n_class)

    def forward(self, x):
        x = self.to_patch(x).flatten(2).transpose(1, 2)                           # (枚数, 64, dim)：パッチの列
        x = torch.cat([self.cls.expand(len(x), -1, -1), x], dim=1) + self.pos
        return self.head(self.encoder(x)[:, 0])

## (3) MLP・CNN・ViT をシード 0〜4 で回す（全部で 20〜60 秒）

In [ ]:
import pandas as pd
from common.device import get_device
from common.logger import ResultLogger
from common.mydata import image_dir, make_model, run, diff_ratio, verdict

device = get_device()
IMAGE_DIR = image_dir()
logger = ResultLogger(None, f"member{MEMBER_NO}", course="c1", day="d4", exercise="opt4", device=device)
rows = []
for name, kind in [("MLP", "MLP"), ("CNN", "CNN"), ("ViT", TinyViT)]:
    for seed in range(5):
        r = run(kind, seed=seed, root=IMAGE_DIR, device=device)
        logger.log_many({k: r[k] for k in ["test_acc", "val_acc_best", "train_sec"]}, condition=f"{IMAGE_DIR.name}_{name}", seed=seed)
        rows.append({"モデル": name, "シード": seed, "テスト正解率": r["test_acc"] * 100, "学習時間(秒)": r["train_sec"]})
    print(f"{name} 終了")
df = pd.DataFrame(rows)
n_class = len([d for d in IMAGE_DIR.iterdir() if d.is_dir()])
n_params = {name: sum(p.numel() for p in make_model(kind, n_class).parameters()) for name, kind in [("MLP", "MLP"), ("CNN", "CNN"), ("ViT", TinyViT)]}
print(df.groupby("モデル")[["テスト正解率", "学習時間(秒)"]].agg(["mean", "std"]).round(1).to_string())
print("パラメータ数：", "｜".join(f"{k} {v:,}" for k, v in n_params.items()))
t = df.pivot(index="シード", columns="モデル", values="テスト正解率")
for a, b in [("CNN", "ViT"), ("MLP", "ViT")]:
    d, s, q = diff_ratio(t[a], t[b])
    print(f"{a} → {b}：差 {d:+.1f} ÷ 標準偏差 {s:.1f} = {q:.1f} → {verdict(q)}")

## (4) 考察（任意で提出）

1. 予測と比べて，当たったこと・外れたこと：＿＿
2. ViT が CNN に勝てなかった（または勝った）のはなぜか．CNN が持っていて ViT が持っていない前提に注目して書く：＿＿
3. 写真が 10 倍・1000 倍あったら，順位はどうなると思うか：＿＿